# 🎬 Movie Recommendation System
### VeloxCode Agency Internship Project
**Author:** Maryam | **Level:** Beginner | **Type:** Recommender System

---
## 🎯 Objective
Build a **Netflix-style Movie Recommendation Engine** using:
1. **Content-Based Filtering** — recommends movies similar to what you like
2. **Collaborative Filtering** — recommends movies liked by similar users

### Techniques Used:
- Cosine Similarity
- Pearson Correlation
- User-Item Matrix
- TF-IDF Vectorization

## 📥 Step 1: Install & Import Libraries

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## 📊 Step 2: Create Dataset

In [ ]:
# ---- MOVIES DATASET ----
movies_data = {
    'movie_id': range(1, 21),
    'title': [
        'The Dark Knight', 'Inception', 'Interstellar', 'The Matrix',
        'Avengers Endgame', 'Iron Man', 'Spider-Man', 'Thor Ragnarok',
        'The Godfather', 'Pulp Fiction', 'Forrest Gump', 'The Shawshank Redemption',
        'Titanic', 'The Notebook', 'Pride and Prejudice', 'La La Land',
        'The Conjuring', 'It', 'Get Out', 'A Quiet Place'
    ],
    'genres': [
        'Action Crime Drama', 'Action Sci-Fi Thriller', 'Sci-Fi Drama Adventure', 'Sci-Fi Action',
        'Action Adventure Sci-Fi', 'Action Sci-Fi', 'Action Adventure', 'Action Comedy Fantasy',
        'Crime Drama', 'Crime Drama Thriller', 'Drama Romance', 'Drama',
        'Romance Drama', 'Romance Drama', 'Romance Drama', 'Romance Drama Music',
        'Horror Thriller', 'Horror', 'Horror Thriller', 'Horror Thriller'
    ],
    'director': [
        'Christopher Nolan', 'Christopher Nolan', 'Christopher Nolan', 'Wachowski',
        'Russo Brothers', 'Jon Favreau', 'Sam Raimi', 'Taika Waititi',
        'Francis Coppola', 'Quentin Tarantino', 'Robert Zemeckis', 'Frank Darabont',
        'James Cameron', 'Nick Cassavetes', 'Joe Wright', 'Damien Chazelle',
        'James Wan', 'Andy Muschietti', 'Jordan Peele', 'John Krasinski'
    ],
    'rating': [9.0, 8.8, 8.6, 8.7, 8.4, 7.9, 7.4, 7.9, 9.2, 8.9, 8.8, 9.3, 7.8, 7.9, 7.8, 8.0, 7.5, 7.3, 7.7, 7.5]
}

movies_df = pd.DataFrame(movies_data)
print('🎬 Movies Dataset:')
print(f'Shape: {movies_df.shape}')
movies_df.head(10)

In [ ]:
# ---- RATINGS DATASET (User-Item Matrix) ----
np.random.seed(42)
n_users = 10
n_movies = 20

# Simulate realistic ratings (0 = not rated)
ratings_matrix = np.zeros((n_users, n_movies))
for user in range(n_users):
    # Each user rates ~60% of movies
    rated_movies = np.random.choice(n_movies, size=12, replace=False)
    for movie in rated_movies:
        ratings_matrix[user, movie] = np.random.randint(1, 6)

# Add some strong preferences per user
# User 0 loves Nolan films (movies 0,1,2)
ratings_matrix[0, 0:3] = 5
# User 1 loves Marvel (movies 4,5,6,7)
ratings_matrix[1, 4:8] = 5
# User 2 loves Romance (movies 12,13,14,15)
ratings_matrix[2, 12:16] = 5

user_ids = [f'User_{i+1}' for i in range(n_users)]
ratings_df = pd.DataFrame(ratings_matrix, index=user_ids, columns=movies_df['title'])

print('👥 User-Item Ratings Matrix:')
print(f'Shape: {ratings_df.shape}')
ratings_df.head()

## 🔍 Step 3: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Movie Dataset - EDA', fontsize=14, fontweight='bold')

# 1. Rating Distribution
axes[0].hist(movies_df['rating'], bins=10, color='gold', edgecolor='black')
axes[0].set_title('Movie Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# 2. Genre Distribution
all_genres = []
for g in movies_df['genres']:
    all_genres.extend(g.split())
genre_series = pd.Series(all_genres).value_counts().head(8)
axes[1].barh(genre_series.index, genre_series.values, color='steelblue')
axes[1].set_title('Genre Frequency')
axes[1].set_xlabel('Count')

# 3. Ratings Heatmap (first 10 movies for clarity)
subset = ratings_df.iloc[:, :10]
im = axes[2].imshow(subset.values, cmap='YlOrRd', aspect='auto')
axes[2].set_title('User-Item Matrix (Subset)')
axes[2].set_xlabel('Movie Index')
axes[2].set_ylabel('User')
axes[2].set_yticks(range(len(user_ids)))
axes[2].set_yticklabels(user_ids, fontsize=8)
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.savefig('eda_movies.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved!')

## 🎯 Step 4: Content-Based Filtering
Recommend movies based on **genres and director similarity** using TF-IDF + Cosine Similarity

In [ ]:
# Combine genres + director into a single text feature
movies_df['content_features'] = movies_df['genres'] + ' ' + movies_df['director']

# TF-IDF Vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_df['content_features'])

print('TF-IDF Matrix Shape:', tfidf_matrix.shape)

# Compute Cosine Similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print('Cosine Similarity Matrix Shape:', cosine_sim.shape)

In [ ]:
def content_based_recommend(movie_title, n=5):
    """
    Recommend movies similar to the given movie title
    using content-based filtering (TF-IDF + Cosine Similarity)
    """
    # Find movie index
    if movie_title not in movies_df['title'].values:
        return f'❌ Movie "{movie_title}" not found in dataset.'

    idx = movies_df[movies_df['title'] == movie_title].index[0]

    # Get similarity scores for all movies
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity (descending), exclude self
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx]

    # Get top-N recommendations
    top_indices = [s[0] for s in sim_scores[:n]]
    top_scores  = [round(s[1], 4) for s in sim_scores[:n]]

    recommendations = movies_df.iloc[top_indices][['title', 'genres', 'rating']].copy()
    recommendations['similarity_score'] = top_scores
    return recommendations

# Test it
print('🎬 Content-Based Recommendations for: "Inception"')
print('='*60)
content_based_recommend('Inception', n=5)

In [ ]:
# Visualize similarity matrix for first 10 movies
plt.figure(figsize=(10, 8))
subset_sim = cosine_sim[:10, :10]
labels = movies_df['title'].values[:10]
short_labels = [t[:15] + '...' if len(t) > 15 else t for t in labels]

sns.heatmap(subset_sim, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=short_labels, yticklabels=short_labels)
plt.title('Content-Based Cosine Similarity Matrix (First 10 Movies)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('cosine_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

## 👥 Step 5: Collaborative Filtering
Recommend movies based on **similar users' ratings** using Pearson Correlation

In [ ]:
# User-User Collaborative Filtering with Pearson Correlation
# Replace 0s with NaN for proper correlation calculation
ratings_for_corr = ratings_df.replace(0, np.nan)

# Compute user-user correlation
user_correlation = ratings_for_corr.T.corr(method='pearson')

print('User Correlation Matrix:')
user_correlation.head()

In [ ]:
def collaborative_recommend(target_user, n_movies=5, n_similar_users=3):
    """
    Recommend movies for a user based on similar users' ratings (Pearson Correlation)
    """
    if target_user not in user_correlation.index:
        return f'❌ User "{target_user}" not found.'

    # Find most similar users (excluding self)
    similar_users = user_correlation[target_user].drop(target_user).nlargest(n_similar_users)
    print(f'👥 Most Similar Users to {target_user}:')
    for user, corr in similar_users.items():
        print(f'   {user}: correlation = {corr:.4f}')

    # Movies already rated by target user
    target_rated = set(ratings_df.columns[ratings_df.loc[target_user] > 0])

    # Aggregate ratings from similar users for unrated movies
    rec_scores = {}
    for sim_user, corr_score in similar_users.items():
        if corr_score <= 0:
            continue
        sim_ratings = ratings_df.loc[sim_user]
        for movie, rating in sim_ratings.items():
            if movie not in target_rated and rating > 0:
                if movie not in rec_scores:
                    rec_scores[movie] = 0
                rec_scores[movie] += corr_score * rating

    # Sort by score
    rec_df = pd.DataFrame(list(rec_scores.items()), columns=['title', 'score'])
    rec_df = rec_df.sort_values('score', ascending=False).head(n_movies)
    rec_df = rec_df.merge(movies_df[['title', 'genres', 'rating']], on='title')
    rec_df['score'] = rec_df['score'].round(4)
    return rec_df

print('\n🎬 Collaborative Filtering Recommendations for: User_1')
print('='*60)
collaborative_recommend('User_1', n_movies=5)

In [ ]:
# Visualize user correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(user_correlation, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1)
plt.title('User-User Pearson Correlation Matrix', fontsize=12)
plt.tight_layout()
plt.savefig('user_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔀 Step 6: Hybrid Recommendation (Content + Collaborative)

In [ ]:
def hybrid_recommend(user_id, liked_movie, n=5):
    """
    Hybrid: combines content-based and collaborative filtering
    """
    print(f'🔀 Hybrid Recommendations for {user_id} who liked "{liked_movie}"')
    print('='*65)

    # Content-based candidates
    cb_recs = content_based_recommend(liked_movie, n=10)
    if isinstance(cb_recs, str):
        print(cb_recs)
        return

    cb_titles = set(cb_recs['title'].values)

    # Filter: keep only movies NOT yet rated by user
    user_rated = set(ratings_df.columns[ratings_df.loc[user_id] > 0]) if user_id in ratings_df.index else set()
    cb_unrated = cb_titles - user_rated

    final_recs = movies_df[movies_df['title'].isin(cb_unrated)][['title', 'genres', 'rating']]
    final_recs = final_recs.sort_values('rating', ascending=False).head(n)

    print(final_recs.to_string(index=False))
    return final_recs

hybrid_recommend('User_2', 'Iron Man', n=5)

## 📊 Step 7: Evaluation

In [ ]:
# Precision@K evaluation for collaborative filtering
def precision_at_k(user_id, k=5, threshold=3.5):
    """
    Precision@K: what fraction of top-K recommendations
    are actually relevant (rated >= threshold) by the user?
    """
    # Split: hide some of user's ratings for evaluation
    user_ratings = ratings_df.loc[user_id]
    rated_movies = user_ratings[user_ratings > 0]

    if len(rated_movies) < 4:
        return None

    # Hide last 3 ratings as 'test'
    test_movies = set(rated_movies.index[-3:])
    relevant = {m for m in test_movies if user_ratings[m] >= threshold}

    # Get recommendations
    recs = collaborative_recommend(user_id, n_movies=k)
    if isinstance(recs, str):
        return None
    rec_titles = set(recs['title'].values)

    hits = len(rec_titles & relevant)
    precision = hits / k if k > 0 else 0
    return precision

# Evaluate for all users
print('📊 Precision@5 Evaluation:')
print('='*40)
precisions = []
for user in user_ids:
    p = precision_at_k(user, k=5)
    if p is not None:
        precisions.append(p)
        print(f'{user}: Precision@5 = {p:.2f}')

print(f'\n🏆 Average Precision@5: {np.mean(precisions):.4f}')

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Recommendation System Summary', fontsize=14, fontweight='bold')

# 1. Top rated movies
top_movies = movies_df.nlargest(8, 'rating')
axes[0].barh(top_movies['title'], top_movies['rating'], color='gold', edgecolor='black')
axes[0].set_title('Top 8 Highest Rated Movies')
axes[0].set_xlabel('Rating')
axes[0].invert_yaxis()

# 2. Ratings sparsity heatmap
binary_matrix = (ratings_df > 0).astype(int)
sns.heatmap(binary_matrix, cmap='Blues', ax=axes[1], cbar=False,
            xticklabels=False)
axes[1].set_title('Ratings Sparsity (Blue = Rated)')
axes[1].set_xlabel('Movies')
axes[1].set_ylabel('Users')

plt.tight_layout()
plt.savefig('recommendation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Summary plots saved!')

## ✅ Step 8: Conclusion

| Approach | Method | Strength |
|----------|--------|----------|
| Content-Based | TF-IDF + Cosine Similarity | Works without user history |
| Collaborative | Pearson Correlation | Captures user preferences |
| Hybrid | Content + Collaborative | Best of both worlds |

### 🏆 Key Takeaways:
- **Content-Based** filtering uses movie features (genres, director) to find similar movies
- **Collaborative Filtering** uses user behavior patterns and Pearson correlation
- **Cosine Similarity** measures angle between feature vectors (0 = dissimilar, 1 = identical)
- **Hybrid systems** (like Netflix) combine both for best accuracy
- **Cold Start Problem**: new users/movies have limited data for collaborative filtering

---
*Project by Maryam | VeloxCode Agency Internship*